In [1]:
# TRAINING DEEP LEARNING LANGUAGE MODEL
"""
Here model will learn deep contextual meaning of text
using pretrained transformer models from HuggingFace
"""

'\nHere model will learn deep contextual meaning of text\nusing pretrained transformer models from HuggingFace\n'

In [2]:
import pandas as pd
import numpy as np
import torch
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

#HuggingFace Transformers
from transformers import (
    AutoTokenizer, #convert text -> tokens/numbers
    AutoModelForSequenceClassification, #pretrained transformer model(BERT, RoBERTa)
    TrainingArguments, # define training settings
    Trainer #high level API to train/evaluate model
)

In [3]:
from datasets import Dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
"""
Prepared a Transformer based deep learning pipeline
1. Pytorch - deep learning framework
2. Transformers - Pretrained Lnaguage Model
3. Dataset - Data formatiing
4. Trainer - training automation
"""

'\nPrepared a Transformer based deep learning pipeline\n1. Pytorch - deep learning framework\n2. Transformers - Pretrained Lnaguage Model\n3. Dataset - Data formatiing\n4. Trainer - training automation\n'

In [5]:
# If uploaded directly
data = pd.read_csv("FINAL_DATA.csv")


In [6]:
#data = pd.read_csv("../data/FINAL_DATA.csv")
X = data['Clean_Reviews'].astype(str).tolist()
y = data['Sentiment_label'].astype(int).tolist()

X_train, X_temp, y_train, y_temp = train_test_split(X, y,
                                                    test_size = 0.3,
                                                    random_state = 42,
                                                    stratify = y)
# 70% training data + 30% temporary data for testing and validation

X_val, X_test, y_val, y_test = train_test_split(X_temp,y_temp,
                                                test_size=0.5,
                                                random_state=42,
                                                stratify=y_temp)
# now 30% temp is split into 15 % testing and 15% validation data

"""
Training Data : model learning
Validation Data : Hyperparameter tuning
Test Data : final evaluation
"""

print(f"1. Length of training data: {len(X_train)}\n2. Length of validation data: {len(X_val)}\n3. Lentgh of test data: {len(X_test)}")



1. Length of training data: 14639
2. Length of validation data: 3137
3. Lentgh of test data: 3137


In [8]:
"""
Training and Evaluating a pretrained transformer model for sentiment analysis
- tokenizes test
- convert data into model format numbers
- fine tune the pretrained model
- evaluating the performance then on test data
"""

def train_model(model_name, X_train, X_val, X_test, y_train, y_val, y_test): # pass models like BERT, RoBERTa etc
    """
    This function will fine tune a pretrained transformer model using HUGGING FACE Trainer by tokenizing input text,
    training on labeled sentiment data -> then results: Accuracy and F1 Score
    """
    print(f"Training Transformer Model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name) # load pretrained tokenizer for the specific model

    def clean_texts(texts):
        return ["empty" if str(x).strip() == ""
                else str(x).strip() for x in texts]

    X_train = clean_texts(X_train)
    X_val   = clean_texts(X_val)
    X_test  = clean_texts(X_test)

    def encode(texts, labels): #ocnvert the raw text into HUgging Face data format
        encodings = tokenizer(list(texts),
                              truncation=True, #cuts long sentences
                              padding = True, # each sentence of equal length
                              max_length = 128 # maxno. of token in each sentence
                              )
        encodings['labels'] = list(labels)
        return Dataset.from_dict(encodings) #convert to huggingface dataset

    train_ds = encode(X_train, y_train)
    test_ds = encode(X_test, y_test)
    val_ds = encode(X_val, y_val)

    #load pretrained transformer
    model = AutoModelForSequenceClassification.from_pretrained(model_name,
                                                               num_labels = 3,  #negative, neutral, positve
                                                               ignore_mismatched_sizes=True
                                                               )
    # performance during training
    def evaluation_metrics(eval_pred):
        logits, labels = eval_pred #tuple upack
        preds = np.argmax(logits, axis=1) # model output predicted probs -> classes
        acc = accuracy_score(labels, preds)
        f1 = f1_score(labels, preds, average='macro')

        return {'accuracy': acc,
                'f1_macro': f1}

    # setting up trasnformers training process
    # 1. how the model should be tarined
    args = TrainingArguments(
        output_dir=f"../results/{model_name.split('/')[-1]}", # save train model result
        num_train_epochs=1,
        per_device_train_batch_size=8, #during training
        per_device_eval_batch_size=32, # during evaluation
        eval_strategy="epoch",  #evaluate after every epoch
        save_strategy="epoch",  # save the model after every epoch
        logging_steps=50,
        fp16=True,
        load_best_model_at_end=True,  # keep best model
        metric_for_best_model="f1_macro", # select the best model based on f1 score
        report_to="none"
    )

    #2. What to train and whch data
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=evaluation_metrics
    )

    trainer.train()
    results = trainer.predict(test_ds)
    print("Results: ", results)
    return results




In [9]:
"""
Now train the models (TRANSORMER EXPERIMENTS)
Goal: finding the pretrained model that works best for multilingual/ Roman urdu sentiment anaylsis

1. BERT - Bidirectional encoder representations from transformers
 - learn conetxtual meaning of words
 - trained over 100 languages
2. XLM - RoBERT
 - trained on huge multilingual data
3. MuRIL - Multilingual representation for Indian languages
 - google specially tarined model for the south asian languages
 - code mixed text
 - closet to my domain

"""

#1. BERT
bert_Results = train_model("bert-base-multilingual-cased",
                            X_train, X_val, X_test,
                            y_train, y_val, y_test)

#2. XML
xlm_results = train_model("xlm-roberta-base",
                            X_train, X_val, X_test,
                            y_train, y_val, y_test)



Training Transformer Model: bert-base-multilingual-cased


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.347929,0.457575,0.839018,0.766491


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Results:  PredictionOutput(predictions=array([[ 0.25854492, -0.26123047, -0.13867188],
       [-1.6464844 , -1.0351562 ,  2.7597656 ],
       [-2.4296875 , -1.2109375 ,  3.6738281 ],
       ...,
       [-1.4736328 , -1.015625  ,  2.5898438 ],
       [-2.2949219 , -1.1943359 ,  3.5371094 ],
       [-1.5234375 , -1.0322266 ,  2.6367188 ]], dtype=float32), label_ids=array([0, 2, 2, ..., 2, 2, 2]), metrics={'test_loss': 0.4279833436012268, 'test_accuracy': 0.8514504303474657, 'test_f1_macro': 0.7819733004573776, 'test_runtime': 5.1915, 'test_samples_per_second': 604.253, 'test_steps_per_second': 19.069})


In [11]:
# 3. MuRIL
muril_results = train_model("google/muril-base-cased",
                            X_train, X_val, X_test,
                            y_train, y_val, y_test)

Training Transformer Model: google/muril-base-cased


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.437656,0.473554,0.836149,0.755726


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Results:  PredictionOutput(predictions=array([[ 1.4882812e+00,  6.0653687e-04, -1.4169922e+00],
       [-1.9062500e+00, -9.7802734e-01,  2.6406250e+00],
       [-1.9042969e+00, -9.8974609e-01,  2.6484375e+00],
       ...,
       [-1.8994141e+00, -9.2578125e-01,  2.5859375e+00],
       [-1.9042969e+00, -9.8388672e-01,  2.6445312e+00],
       [-1.9042969e+00, -9.7851562e-01,  2.6386719e+00]], dtype=float32), label_ids=array([0, 2, 2, ..., 2, 2, 2]), metrics={'test_loss': 0.45205095410346985, 'test_accuracy': 0.8527255339496335, 'test_f1_macro': 0.7807544942404578, 'test_runtime': 5.6415, 'test_samples_per_second': 556.058, 'test_steps_per_second': 17.549})


In [ ]:
results = {
    "mBERT": bert_Results,
    "XLM-RoBERTa": xlm_results,
    "MuRIL": muril_results
}

# as the results contain numpy arrays and numpy numbers like float so json cant save them directly -> causing errors
# so using customer json encoder

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray): # convert array to list
            return obj.tolist()
        if isinstance(obj, (np.integer, np.floating, np.bool_)):
            return obj.item()  # converts any numpy scalar to normal Python type (n.float32(0.56) -> 0.56)
        return super().default(obj)

with open("..results/transformer_results.json", "w") as f:
    json.dump(results, f, indent=2, cls=NumpyEncoder)

print("Saved!")


Saved!


In [7]:

test_texts = list(X_test)
test_labels = list(y_test)

test_df = pd.DataFrame({
    "Text": test_texts,
    "true_label": test_labels
})

test_df.to_csv("../results/transformer_test_set.csv", index=False)